# 02 — Polysemanticity & SAE Features

Load the latest `polysemanticity_sae` run, inspect the feature dictionary,
visualise dead vs live feature distribution, and discuss top-K sparsity and
coherence scores.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

repo_root = Path("__file__").parent.parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

In [ ]:
from mech_interp.storage.sqlite_store import SQLiteResultStore

store = SQLiteResultStore(
    database_path=repo_root / "artifacts" / "results.db",
    artifact_dir=repo_root / "artifacts",
)

runs = store.list_runs(limit=100)
sae_runs = [r for r in runs if r.family == "polysemanticity_sae"]

if not sae_runs:
    raise RuntimeError(
        "No polysemanticity_sae runs found. "
        "Run: mech run --name polysemanticity_sae"
    )

latest = sae_runs[0]
result = store.get_result(latest.id)
print(f"Run {latest.id} | {latest.spec_name} | status={latest.status}")
print("Metrics:", json.dumps(result.metrics if result else {}, indent=2))

## Top-20 features by max_activation

The SAE encodes residual-stream activations into a wider sparse dictionary.
Each "feature" is a learnt direction in activation space.  `max_activation`
is the largest value that feature achieved across the training corpus —
a proxy for how strongly it fires.

In [ ]:
if result is None:
    raise RuntimeError(f"No result stored for run {latest.id}")

features_path = result.artifacts.get("features_json")
if features_path:
    features_data = json.loads(Path(features_path).read_text())
    features = features_data if isinstance(features_data, list) else features_data.get("features", [])
    df_features = pd.DataFrame(features)
    if "max_activation" in df_features.columns:
        top20 = df_features.nlargest(20, "max_activation")[["feature_id", "max_activation", "dead"] if "dead" in df_features.columns else ["feature_id", "max_activation"]]
        print(top20.to_string(index=False))
    else:
        print(df_features.head(20).to_string(index=False))
else:
    # Reconstruct from metrics.
    metrics = result.metrics
    summary = {
        "total_features": metrics.get("feature_count", metrics.get("dictionary_size", "?") ),
        "dead_features": metrics.get("dead_features", "?"),
        "live_features": metrics.get("live_features", "?"),
        "mean_sparsity": metrics.get("mean_sparsity", metrics.get("avg_active_features", "?")),
    }
    print("Feature summary (from metrics):")
    for k, v in summary.items():
        print(f"  {k}: {v}")

## Dead vs live feature distribution

A "dead" feature never fires above the threshold on the training corpus.
Dead features indicate the SAE is over-parameterised for the observed
activation distribution — a common early-training artefact.

In [ ]:
metrics = result.metrics

dead = float(metrics.get("dead_features", 0))
live = float(metrics.get("live_features", metrics.get("feature_count", 1)) ) - dead
if live < 0:
    live = 0.0

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["live", "dead"], [live, dead], color=["steelblue", "tomato"])
ax.set_ylabel("Feature count")
ax.set_title("SAE feature liveness")
for i, v in enumerate([live, dead]):
    ax.text(i, v + 0.5, str(int(v)), ha="center")
plt.tight_layout()
plt.show()

## Top-activating prompts for feature 0

For each feature we store the prompts that caused the highest activation.
Inspecting these side-by-side is the primary way to build intuitions about
what concept a feature represents.

In [ ]:
top_prompts_path = result.artifacts.get("top_activating_prompts")
if top_prompts_path and Path(top_prompts_path).exists():
    top_data = json.loads(Path(top_prompts_path).read_text())
    # Expect list of {feature_id, prompts: [...]} or flat list.
    if isinstance(top_data, list) and top_data and "prompts" in top_data[0]:
        feature_entry = top_data[0]
        print(f"Feature {feature_entry.get('feature_id', 0)} top prompts:")
        for i, p in enumerate(feature_entry["prompts"][:5], 1):
            print(f"  [{i}] {p}")
    else:
        print(str(top_data)[:400])
else:
    print("top_activating_prompts artifact not found — run with a larger corpus to generate it.")

## Top-K sparsity and coherence

**Top-K sparsity**: on any given token, only K of the D dictionary features
are active.  Low K means the representation is highly compressed.  The SAE
is trained to minimise reconstruction loss subject to this sparsity penalty.

**Coherence score**: a measure borrowed from topic modelling.  For a feature
to be "coherent", its top-activating prompts should share a semantic theme.
Coherence ≈ 1 means the feature cleanly represents one concept;
coherence ≈ 0 means it fires for many unrelated contexts — the classic
polysemantic pattern that motivates SAE decomposition in the first place.